# 02 · word2vec: CBOW, Skip-gram e a aritmética de vetores

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


**Artigo-base (leitura obrigatória):** Mikolov, Chen, Corrado & Dean (2013a), *Efficient Estimation of Word Representations in Vector Space*.

## O que o artigo propôs
Até 2013, os melhores vetores vinham de **modelos neurais de linguagem** (Bengio et al., 2003; Collobert & Weston, 2008): redes com camada oculta não linear, caras de treinar (dias para poucas centenas de milhões de palavras).
Mikolov et al. fazem uma pergunta de engenharia: *quanto da qualidade dos vetores sobrevive se tirarmos a camada oculta?* A resposta são duas arquiteturas **log-lineares**:

* **CBOW** (*continuous bag-of-words*): prevê a palavra central a partir da **média** dos vetores do contexto (a ordem é ignorada — daí "bag").
* **Skip-gram**: prevê cada palavra do contexto a partir da palavra central.

Complexidade por exemplo de treino (Tabela do artigo, §3): CBOW $O(N\cdot D + D\cdot\log_2 V)$ e Skip-gram $O(C\cdot(D + D\cdot\log_2 V))$, contra $O(N\cdot D + N\cdot D\cdot H + H\cdot\log_2 V)$ do NNLM com camada oculta $H$. O termo $\log_2 V$ vem do **softmax hierárquico** (notebook 03).

Além da eficiência, o artigo cunhou o teste de **analogias**: `vec("king") − vec("man") + vec("woman") ≈ vec("queen")`, com 19.544 questões em 14 categorias (5 semânticas, 9 sintáticas) — o `questions-words.txt`.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: /home/user/linguagem-blis2s26 | fast: False


## 1. Premissas: o modelo log-linear e o softmax

Para a palavra central $w_I$ e uma palavra de contexto $w_O$, o Skip-gram define

$$p(w_O \mid w_I) = \frac{\exp(v'_{w_O}{}^{\top} v_{w_I})}{\sum_{w=1}^{V} \exp(v'_{w}{}^{\top} v_{w_I})}$$

com **dois** vetores por palavra: $v_w$ ("entrada", matriz $W_{in}$) e $v'_w$ ("saída", $W_{out}$). O objetivo é maximizar a log-verossimilhança média dos contextos numa janela de tamanho $c$:

$$\frac{1}{T}\sum_{t=1}^{T}\sum_{-c\le j\le c,\ j\ne 0} \log p(w_{t+j}\mid w_t).$$

"Log-linear" = o logaritmo da probabilidade é linear nos parâmetros ($v'^{\top}v$): não há não linearidade entre entrada e saída. É isso que torna o modelo rápido **e** que explica a aritmética de vetores (Mikolov 2013b, §5): somar vetores ≈ multiplicar distribuições de contexto.

O denominador custa $O(V)$ por exemplo — inviável para $V\sim 10^5$–$10^6$. O artigo usa softmax hierárquico; o 2013b introduz a amostragem negativa (ambos no notebook 03). Aqui usamos amostragem negativa por padrão.

## 2. Implementação *from scratch* (numpy)

A classe `replang.models.word2vec_np.Word2Vec` espelha os parâmetros do `word2vec.c`: `sg` (1 = Skip-gram, 0 = CBOW), `window`, `negative`, `hs`, `sample`, `alpha`.
Vamos abrir o capô: vocabulário, pares (centro, contexto), passo de gradiente.

In [2]:
import inspect
from replang.models.word2vec_np import Word2Vec, Vocab
from replang.data.corpora import toy_corpus
toy = toy_corpus() * 20   # repetimos para ter estatística suficiente
model = Word2Vec(dim=20, window=2, sg=1, negative=5, sample=0, epochs=1, seed=1).build_vocab(toy)
print("|V| =", len(model.vocab), "| 10 mais frequentes:", model.vocab.words[:10])
enc = model._encode_corpus(toy[:1])
pares = list(model._pairs(enc))
print("sentença:", toy[0]); print("pares (centro → contexto), janela efetiva sorteada em [1, window]:")
print([(model.vocab.words[a], model.vocab.words[b]) for a, b in pares][:12])

|V| = 57 | 10 mais frequentes: ['a', 'o', 'e', 'é', 'come', 'rei', 'rainha', 'com', 'da', 'gato']
sentença: ['o', 'rei', 'governa', 'o', 'reino', 'com', 'a', 'rainha']
pares (centro → contexto), janela efetiva sorteada em [1, window]:
[('o', 'rei'), ('rei', 'o'), ('rei', 'governa'), ('governa', 'o'), ('governa', 'rei'), ('governa', 'o'), ('governa', 'reino'), ('o', 'rei'), ('o', 'governa'), ('o', 'reino'), ('o', 'com'), ('reino', 'o')]


In [3]:
# O passo de gradiente da amostragem negativa (explicado em detalhe no notebook 03)
print(inspect.getsource(Word2Vec._step_neg))

    def _step_neg(self, h: np.ndarray, targets: np.ndarray, lr: float) -> tuple[float, np.ndarray]:
        """Amostragem negativa (eq. 4 de Mikolov 2013b).

        ``h``: (B, d) representação de entrada (vetor da palavra central no SG; média do contexto no CBOW);
        ``targets``: (B,) palavra a prever. Retorna perda média e gradiente em relação a ``h``.
        """
        B = len(targets)
        neg = self._sample_negatives((B, self.negative))
        # (B, 1+k, d): positivo seguido de k negativos
        idx = np.concatenate([targets[:, None], neg], axis=1)
        out = self.W_out[idx]
        labels = np.zeros((B, 1 + self.negative), dtype=np.float32)
        labels[:, 0] = 1.0
        score = np.einsum("bd,bkd->bk", h, out)
        p = sigmoid(score)
        loss = -(np.log(p[:, 0] + 1e-9).sum() + np.log(1 - p[:, 1:] + 1e-9).sum()) / B
        g = (p - labels) * lr  # dL/dscore
        grad_h = np.einsum("bk,bkd->bd", g, out)
        grad_out = np.einsum("bk,bd->bkd", g, h)

### 2.1 Treinando Skip-gram e CBOW no toy corpus

Observe a **perda** cair e os vizinhos de `rei` se organizarem. A perda da amostragem negativa é
$-\log\sigma(v'_{w_O}{}^{\top}v_{w_I}) - \sum_{i=1}^{k}\log\sigma(-v'_{w_i}{}^{\top}v_{w_I})$; com $k=5$ negativos o valor inicial é $\approx 6\log 2 \approx 4{,}16$.

In [4]:
from replang.viz import training_curve, scatter_words
sg = Word2Vec(dim=20, window=2, sg=1, negative=5, sample=0, epochs=25, batch_size=64, seed=1).train(toy)
cb = Word2Vec(dim=20, window=2, sg=0, negative=5, sample=0, epochs=25, batch_size=64, seed=1).train(toy)
hist = pd.concat([pd.DataFrame(sg.history).assign(modelo="Skip-gram"), pd.DataFrame(cb.history).assign(modelo="CBOW")])
import plotly.express as px
px.line(hist, x="epoch", y="loss", color="modelo", markers=True, template="plotly_white", title="Perda média por época (toy corpus)").show()
wv_sg, wv_cb = sg.to_wordvectors("SG toy"), cb.to_wordvectors("CBOW toy")
for w in ["rei", "gato", "paris"]:
    print(f"{w:<6} SG: {[x for x, _ in wv_sg.most_similar(w, topn=4)]}   CBOW: {[x for x, _ in wv_cb.most_similar(w, topn=4)]}")

rei    SG: ['menino', 'reino', 'homem', 'rainha']   CBOW: ['menino', 'príncipe', 'homem', 'cachorro']
gato   SG: ['cachorro', 'sofá', 'tapete', 'menino']   CBOW: ['cachorro', 'menino', 'mia', 'reino']
paris  SG: ['roma', 'capital', 'lisboa', 'madri']   CBOW: ['roma', 'rainha', 'princesa', 'capital']


In [5]:
scatter_words(wv_sg, {"realeza": ["rei", "rainha", "príncipe", "princesa"], "pessoas": ["homem", "mulher", "menino", "menina"],
                      "animais": ["gato", "cachorro"], "frutas": ["maçã", "banana"], "lugares": ["paris", "lisboa", "roma", "madri", "frança", "portugal", "itália", "espanha"]},
              title="Skip-gram (20d) no toy corpus — PCA", arrows=[("rei", "rainha"), ("homem", "mulher"), ("príncipe", "princesa"), ("menino", "menina")])

As setas `rei→rainha`, `homem→mulher`, `príncipe→princesa`, `menino→menina` são aproximadamente **paralelas**: o "deslocamento de gênero" é quase o mesmo vetor. Esta é a origem geométrica tanto das analogias (Mikolov) quanto da **direção de gênero** de Bolukbasi (notebook 08).

## 3. Entrada × saída: por que dois vetores por palavra?

$W_{in}$ representa a palavra **como alvo**; $W_{out}$ a representa **como contexto**. O produto $v'_c{}^\top v_w$ mede "quão bem $c$ prediz $w$".
Palavras que coocorrem têm $v_w$ parecido com $v'_c$ (relação sintagmática); palavras intercambiáveis têm $v_w$ parecido com $v_{w'}$ (relação paradigmática). O `word2vec.c` descarta $W_{out}$; GloVe soma as duas (notebook 04).

In [6]:
w = "rei"
print("vizinhos em W_in  (paradigmáticos):", [x for x, _ in sg.to_wordvectors(combine="in").most_similar(w, topn=5)])
print("vizinhos em W_out (contextos):      ", [x for x, _ in sg.to_wordvectors(combine="out").most_similar(w, topn=5)])
vin, vout = sg.W_in, sg.W_out
scores = vout @ vin[sg.vocab.index[w]]
p = np.exp(scores - scores.max()); p /= p.sum()
print("p(contexto | rei) via softmax completo:", [(sg.vocab.words[i], round(float(p[i]), 3)) for i in np.argsort(-p)[:6]])

vizinhos em W_in  (paradigmáticos): ['menino', 'reino', 'homem', 'rainha', 'príncipe']
vizinhos em W_out (contextos):       ['reino', 'menino', 'cachorro', 'homem', 'príncipe']
p(contexto | rei) via softmax completo: [('do', 0.233), ('governa', 0.144), ('um', 0.133), ('filho', 0.108), ('o', 0.105), ('come', 0.062)]


## 4. Escalando: Machado de Assis (2,4 M tokens)

A implementação numpy é didática, não rápida. Para o corpus completo usamos o **gensim** (`replang.models.trainers`), que implementa o mesmo algoritmo em C/Cython. Os modelos foram treinados por `uv run replang train` e ficam em `data/models`:
`machado_sg100` (Skip-gram, NEG-10), `machado_cbow100`, `machado_sg100_hs` (softmax hierárquico), `machado_sg50_nosub` (sem subamostragem) e `machado_ft100` (fastText).

In [7]:
from replang.data.embeddings import load_local_model
from replang.models.trainers import list_models
pd.DataFrame(list_models())[["name", "algo", "dim", "epochs", "vocab", "seconds"]] if list_models() else print("rode: uv run replang train")

,name,algo,dim,epochs,vocab,seconds
0,machado_bilm,bilm-jax,NaN,NaN,8004.0,NaN
1,machado_cbow100,word2vec,100.0,5.0,23194.0,6.6
2,machado_ft100,fasttext,100.0,5.0,23194.0,33.6
3,machado_glove100,glove-numpy,100.0,15.0,23194.0,351.0
4,machado_pvdbow100,doc2vec,100.0,20.0,NaN,17.4
5,machado_pvdm100,doc2vec,100.0,20.0,NaN,24.2
6,machado_sg100,word2vec,100.0,5.0,23194.0,20.1
7,machado_sg100_hs,word2vec,100.0,5.0,23194.0,18.0
8,machado_sg50_nosub,word2vec,50.0,5.0,23194.0,36.7


In [8]:
sg_m = load_local_model("machado_sg100"); cb_m = load_local_model("machado_cbow100")
for w in ["capitu", "amor", "escravo", "dinheiro", "triste", "rua"]:
    print(f"{w:<9} SG:   {[x for x, _ in sg_m.most_similar(w, topn=6)]}")
    print(f"{'':<9} CBOW: {[x for x, _ in cb_m.most_similar(w, topn=6)]}")

capitu    SG:   ['lalau', 'ângela', 'augusta', 'mariana', 'amélia', 'espantada']
          CBOW: ['tito', 'repetiu', 'eugênia', 'ângela', 'baronesa', 'carlota']
amor      SG:   ['amante', 'afeto', 'gozo', 'orgulho', 'vínculo', 'fruto']
          CBOW: ['dever', 'sentimento', 'talento', 'caráter', 'interesse', 'autor']
escravo   SG:   ['pádua', 'licenciado', 'respondeu-lhe', 'levara', 'norberto', 'hortelão']
          CBOW: ['convite', 'recado', 'parente', 'perguntou-lhe', 'vizinho', 'primo']
dinheiro  SG:   ['pagar', 'preço', 'emprego', 'ganho', 'ganhar', 'gastar']
          CBOW: ['parecer', 'namoro', 'pagar', 'perdido', 'engano', 'emprego']
triste    SG:   ['abatida', 'alegre', 'aborrecida', 'abatido', 'tranqüila', 'preocupada']
          CBOW: ['alegre', 'estela', 'carlota', 'emília', 'cecília', 'pareceu']
rua       SG:   ['esquina', 'quitanda', 'ouvidor', 'ourives', 'loja', 'ajuda']
          CBOW: ['praia', 'loja', 'chácara', 'esquina', 'sala', 'escada']


## 5. Analogias: o teste de Mikolov et al.

> *"We find that... vector("King") − vector("Man") + vector("Woman") results in a vector that is closest to vector("Queen")."*

O método **3CosAdd** busca $\arg\max_y \cos(y,\ b - a + c)$ excluindo $a, b, c$. O conjunto `questions-words.txt` tem 14 categorias; aqui usamos a versão **PT-BR** (Rodrigues et al., 2016) adotada por Hartmann et al. (2017), e a inglesa com o GloVe 50d para comparação.

In [9]:
from replang.data.analogies import load_analogies
pt_an = load_analogies("pt-br"); en_an = load_analogies("en")
print(f"PT-BR: {pt_an.n:,} questões em {len(pt_an.categories)} categorias | EN: {en_an.n:,} questões")
pt_an.summary()

PT-BR: 17,558 questões em 14 categorias | EN: 19,544 questões


,categoria,tipo,questões,exemplo
0,capital-common-countries,semântica,506,atenas : grécia :: bagdá : iraque
1,capital-world,semântica,4524,abuja : nigéria :: acra : gana
2,currency,semântica,866,argélia : dinar :: angola : kwanza
3,city-in-state,semântica,2467,chicago : ilinóis :: houston : texas
4,family,semântica,462,rapaz : moça :: irmão : irmã
5,gram1-adjective-to-adverb,sintática,930,fantástico : fantasticamente :: aparente : aparentemente
6,gram2-opposite,sintática,756,aceitável : inaceitável :: consciente : inconsciente
7,gram3-comparative,sintática,30,mau : pior :: grande : maior
8,gram4-superlative,sintática,600,mau : pior :: grande : maior
9,gram5-present-participle,sintática,1056,programar : programando :: dançar : dançando


In [10]:
from replang.data.embeddings import load_ptwiki, load_glove_en
pt = load_ptwiki(); en = load_glove_en()
exemplos = [("homem", "rei", "mulher"), ("brasil", "brasília", "frança"), ("bom", "melhor", "ruim"), ("falar", "falando", "comer"), ("rápido", "rapidamente", "feliz"), ("um", "dois", "primeiro")]
for a, b, c in exemplos:
    r = pt.analogy(a, b, c, topn=3) if pt.has(a, b, c) else "OOV"
    print(f"{a} : {b} :: {c} : ?  →  {r}")

homem : rei :: mulher : ?  →  [('rainha', 0.81949782371521), ('consorte', 0.7614757418632507), ('princesa', 0.7310948967933655)]
brasil : brasília :: frança : ?  →  [('toulouse', 0.7129448652267456), ('avinhão', 0.6799794435501099), ('provence', 0.6797589063644409)]
bom : melhor :: ruim : ?  →  [('pior', 0.7648845911026001), ('favorita', 0.6318248510360718), ('péssima', 0.6142615079879761)]
falar : falando :: comer : ?  →  [('comendo', 0.8761519193649292), ('comia', 0.7711364030838013), ('bebendo', 0.7666693329811096)]
rápido : rapidamente :: feliz : ?  →  [('sozinha', 0.6881930232048035), ('finalmente', 0.6834356784820557), ('novamente', 0.6694375276565552)]
um : dois :: primeiro : ?  →  [('primeiros', 0.7191872596740723), ('quatro', 0.7157596349716187), ('três', 0.7102711200714111)]


In [11]:
from replang.eval.analogies import evaluate_analogies
import time
t = time.time()
res_pt = evaluate_analogies(pt, pt_an, restrict=30000, max_per_category=None if not FAST else 300)
print(f"{time.time()-t:.0f}s")
res_pt.style.format({"acurácia": "{:.1%}"}).background_gradient(subset=["acurácia"], cmap="Blues")

1s


,categoria,tipo,n,cobertas,acertos,acurácia
0,capital-common-countries,semântica,506,272,255,93.8%
1,capital-world,semântica,4524,755,582,77.1%
2,currency,semântica,866,106,1,0.9%
3,city-in-state,semântica,2467,855,317,37.1%
4,family,semântica,462,342,211,61.7%
5,gram1-adjective-to-adverb,sintática,930,380,76,20.0%
6,gram2-opposite,sintática,756,240,70,29.2%
7,gram3-comparative,sintática,30,30,12,40.0%
8,gram4-superlative,sintática,600,6,3,50.0%
9,gram5-present-participle,sintática,1056,600,430,71.7%


In [12]:
fig = px.bar(res_pt[~res_pt.categoria.str.startswith("TOTAL")], x="categoria", y="acurácia", color="tipo", template="plotly_white",
             title="Wikipedia2Vec PT 100d — acurácia por categoria (3CosAdd, busca nas 30k palavras mais frequentes)")
fig.update_layout(xaxis_tickangle=-35); fig.show()

### 5.1 O que entra na conta: cobertura, restrição de vocabulário e método

* **Cobertura**: questões com qualquer palavra fora do vocabulário são descartadas (como no `compute-accuracy.c` original, que também restringe a busca às 30 mil palavras mais frequentes). Reportar só a acurácia sem a cobertura é enganoso.
* **3CosMul** (Levy & Goldberg, 2014) costuma dar alguns pontos a mais.
* Modelos treinados em **Machado de Assis** cobrem pouco das categorias (capitais do mundo, moedas) — um corpus literário do século XIX não "sabe" que *Bancoque* é capital da Tailândia. Isso ilustra a dependência do corpus.

In [13]:
rows = []
for name, wv in [("Wikipedia2Vec PT 100d", pt), ("Machado SG 100d", sg_m), ("Machado CBOW 100d", cb_m)]:
    for method in ["add", "mul"]:
        r = evaluate_analogies(wv, pt_an, method=method, restrict=30000, max_per_category=300).set_index("categoria")
        rows.append({"modelo": name, "método": method, "sintática": r.loc["TOTAL sintática", "acurácia"], "semântica": r.loc["TOTAL semântica", "acurácia"],
                     "total": r.loc["TOTAL", "acurácia"], "cobertura": r.loc["TOTAL", "cobertas"] / r.loc["TOTAL", "n"]})
pd.DataFrame(rows).style.format({"sintática": "{:.1%}", "semântica": "{:.1%}", "total": "{:.1%}", "cobertura": "{:.1%}"})

,modelo,método,sintática,semântica,total,cobertura
0,Wikipedia2Vec PT 100d,add,53.8%,60.5%,56.3%,19.1%
1,Wikipedia2Vec PT 100d,mul,51.7%,59.8%,54.8%,19.1%
2,Machado SG 100d,add,2.8%,19.7%,6.0%,12.9%
3,Machado SG 100d,mul,3.0%,20.2%,6.3%,12.9%
4,Machado CBOW 100d,add,0.5%,10.2%,2.4%,12.9%
5,Machado CBOW 100d,mul,0.5%,9.0%,2.1%,12.9%


## 6. Dimensão e tamanho do corpus (Tabela 2 do artigo)

O artigo mostra que aumentar **só** a dimensão ou **só** o corpus satura; é preciso crescer os dois. Reproduzimos a tendência treinando Skip-gram (gensim) em fatias do corpus Machado com dimensões diferentes e medindo analogias **sintáticas** (as semânticas quase não têm cobertura neste corpus).

> Os números absolutos serão **baixos**: um corpus literário do século XIX (2,4 M tokens) é duas a três ordens de grandeza menor que os dos artigos e o teste é "da Wikipédia" (capitais, nacionalidades). O que interessa é a **tendência**: 4× mais tokens → várias vezes mais acertos; dimensão a mais sem dados a mais não ajuda (Tabela 2 do artigo).

In [14]:
from gensim.models import Word2Vec as GWord2Vec
from replang.data.corpora import load_machado_sentences
sents = load_machado_sentences()
grid = [(0.25, 50), (0.25, 100), (1.0, 50), (1.0, 100)] if not FAST else [(0.25, 50), (1.0, 50)]
rows = []
for frac, dim in grid:
    sub = sents[: int(len(sents) * frac)]
    t = time.time()
    m = GWord2Vec(sub, vector_size=dim, window=5, sg=1, negative=10, min_count=5, epochs=3, sample=1e-4, workers=settings.workers, seed=1)
    wv = WordVectors.from_gensim(m.wv)
    r = evaluate_analogies(wv, pt_an, restrict=None).set_index("categoria")
    rows.append({"fração do corpus": frac, "tokens": sum(map(len, sub)), "dim": dim, "sintática": r.loc["TOTAL sintática", "acurácia"],
                 "cobertas": int(r.loc["TOTAL", "cobertas"]), "segundos": round(time.time() - t)})
pd.DataFrame(rows)

,fração do corpus,tokens,dim,sintática,cobertas,segundos
0,0.25,592287,50,0.002747,1302,3
1,0.25,592287,100,0.000916,1302,3
2,1.00,2357192,50,0.011698,3771,13
3,1.00,2357192,100,0.013797,3771,11


## 7. O que o artigo encontrou (resumo dos resultados)

| Modelo (640d, 320 M tokens) | Semântica | Sintática |
|---|---|---|
| RNNLM | 9 % | 36 % |
| NNLM | 23 % | 53 % |
| CBOW | 24 % | 64 % |
| Skip-gram | **55 %** | 59 % |

Com mais dados e dimensões (Skip-gram 1000d, 6 B tokens, DistBelief): 66 % / 65 %. Skip-gram é melhor em semântica; CBOW é mais rápido e competitivo em sintaxe. O treino de um dia em um cluster substituiu semanas do NNLM.

### Pontos para a apresentação
1. A inovação não é "rede neural": é **tirar** a camada oculta e aceitar um modelo mais simples que vê muito mais dados.
2. Dois vetores por palavra; descartar $W_{out}$ é uma escolha, não uma necessidade.
3. A analogia por aritmética é uma **propriedade emergente** do objetivo log-linear — e é exatamente o mecanismo que carrega estereótipos (Bolukbasi).
4. Analogias são um *proxy* barato; o notebook 07 (Hartmann) discute seus limites.

### Perguntas para discussão
* Por que o CBOW "dilui" a semântica? (média do contexto apaga a ordem e as palavras raras)
* O que aconteceria com a analogia `homem : rei :: mulher : ?` se *rainha* fosse uma palavra raríssima no corpus?

## Referências
- MIKOLOV, T.; CHEN, K.; CORRADO, G.; DEAN, J. **Efficient Estimation of Word Representations in Vector Space.** ICLR Workshop, 2013. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- MIKOLOV, T.; SUTSKEVER, I.; CHEN, K.; CORRADO, G.; DEAN, J. **Distributed Representations of Words and Phrases and their Compositionality.** NIPS, 2013. [arXiv:1310.4546](https://arxiv.org/abs/1310.4546)
- LEVY, O.; GOLDBERG, Y. **Neural Word Embedding as Implicit Matrix Factorization.** NIPS, 2014; LEVY, GOLDBERG & DAGAN. **Improving Distributional Similarity with Lessons Learned from Word Embeddings.** TACL, 2015.
- RODRIGUES, J. et al. **LX-DSemVectors: Distributional Semantics Models for Portuguese.** PROPOR, 2016.
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)